In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "ml_df_v4.csv").exists() and (ROOT.parent / "data" / "ml_df_v4.csv").exists():
    ROOT = ROOT.parent

ml_df = pd.read_csv(ROOT / "data" / "ml_df_v4.csv")
if "date" in ml_df.columns:
    ml_df["date"] = pd.to_datetime(ml_df["date"], errors="coerce")

LEAKAGE_COLS = {"home_goals", "away_goals", "score", "result", "home_team_win", "away_team_win", "draw", "extra_time", "penalty_shootout", "score_penalties"}
CATEGORICAL_COLS = ["stage", "tournament_name", "host_country", "home_team", "away_team", "tournament_size_category"]
EXPLICIT_COLS = ["win_rate_diff", "goal_diff_diff", "form_diff", "goals_per_match_diff", "conceded_per_match_diff", "season_win_rate_diff", "season_goal_diff_diff", "elo_diff", "ranking_diff", "wc_goals_diff_before", "total_teams", "matches_played", "goals_scored_tournament", "avg_goals_per_game", "year_normalized", "is_neutral_venue", "home_advantage_strength"]

feature_cols = [c for c in ml_df.columns if c in CATEGORICAL_COLS or c.startswith("home_") or c.startswith("away_") or c in EXPLICIT_COLS]
feature_cols = [c for c in feature_cols if c not in LEAKAGE_COLS and c != "result_target"]
numeric_cols = [c for c in feature_cols if c in ml_df.select_dtypes(include=[np.number]).columns]

In [2]:
# Split by year groups
train_df = ml_df[ml_df["year"] <= 2014]
val_df = ml_df[ml_df["year"] == 2018]
test_df = ml_df[ml_df["year"] == 2022]

def stats_by_group(df, cols):
    d = {}
    for c in cols:
        if c not in df.columns:
            continue
        s = df[c].dropna()
        d[c] = {"mean": s.mean(), "std": s.std(), "pct_missing": (1 - s.count() / len(df)) * 100}
    return pd.DataFrame(d).T

train_stats = stats_by_group(train_df, numeric_cols)
test_stats = stats_by_group(test_df, numeric_cols)

# Flag features where 2022 deviates >2 std from train mean
shifted = []
for c in numeric_cols:
    if c not in train_stats.index or c not in test_stats.index:
        continue
    t_mean, t_std = train_stats.loc[c, "mean"], train_stats.loc[c, "std"]
    s_mean = test_stats.loc[c, "mean"]
    if pd.isna(t_std) or t_std == 0:
        continue
    z = abs(s_mean - t_mean) / t_std if t_std > 0 else 0
    if z > 2:
        shifted.append((c, z, t_mean, s_mean))

shift_df = pd.DataFrame(shifted, columns=["feature", "z_score", "train_mean", "test_mean"]).sort_values("z_score", ascending=False)
print("Features with |z|>2 (2022 vs train):")
print(shift_df.head(15).to_string() if len(shift_df) > 0 else "None")

Features with |z|>2 (2022 vs train):
None


In [3]:
# 2022-specific checks
print("=== 2022 Data Quality ===")
ranking_cols = [c for c in ml_df.columns if "ranking" in c.lower()]
for c in ranking_cols:
    if c in test_df.columns:
        nan_pct = test_df[c].isna().mean() * 100
        print(f"{c}: {nan_pct:.1f}% missing in 2022")

print("\nStage distribution 2018 vs 2022:")
if "stage" in ml_df.columns:
    print(val_df["stage"].value_counts().head(5))
    print(test_df["stage"].value_counts().head(5))

print("\nTeams in 2022 not in train:")
train_teams = set(train_df["home_team"].dropna()) | set(train_df["away_team"].dropna())
test_teams = set(test_df["home_team"].dropna()) | set(test_df["away_team"].dropna())
new_teams = test_teams - train_teams
print(new_teams if new_teams else "None")

=== 2022 Data Quality ===
home_ranking_score: 100.0% missing in 2022
away_ranking_score: 100.0% missing in 2022

Stage distribution 2018 vs 2022:
stage
Group stage       47
Round of 16        7
Quarter-finals     4
Semi-finals        2
Third place        1
Name: count, dtype: int64
stage
Group stage       42
round of 16        8
group stage        6
quarter-finals     4
semi-finals        2
Name: count, dtype: int64

Teams in 2022 not in train:
{'Qatar'}


In [4]:
# Visualization: top shifted features
import matplotlib.pyplot as plt

if len(shift_df) > 0:
    top = shift_df.head(8)["feature"].tolist()
    fig, axes = plt.subplots(2, 4, figsize=(14, 8))
    axes = axes.flatten()
    for i, col in enumerate(top):
        if col not in ml_df.columns:
            continue
        ax = axes[i]
        for name, df in [("train", train_df), ("2022", test_df)]:
            ax.hist(df[col].dropna(), alpha=0.5, label=name, bins=15)
        ax.set_title(col[:25])
        ax.legend(fontsize=8)
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No high-shift features to plot.")

No high-shift features to plot.
